# Task 2 — Localization (§2.5 Object Detection)

**Colab setup:** Runtime → Change runtime type → **T4 GPU**

**Prerequisite:** Task 1 must be done — `classifier.pth` must exist in your Drive.

**After this notebook you will have:**
- `localizer.pth` saved directly to your Google Drive
- W&B detection table logged for §2.5
- `LOCALIZER_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.2a / 4.2b** to Gradescope

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

cls_path = f"{CKPT_DIR}/classifier.pth"
if os.path.exists(cls_path):
    print(f"✅  classifier.pth found ({os.path.getsize(cls_path)/1e6:.1f} MB) — ready to proceed")
else:
    print("❌  classifier.pth NOT found — run Task 1 notebook first!")

## 2 · Setup

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 3 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 4 · §2.5 — Localization Training
Trains bounding-box regressor using the pretrained VGG11 encoder from Task 1.
Loss = MSE + IoU. Detection table logged to W&B at the end.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
!python train.py --task localize --experiment exp-detection --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 --checkpoint-dir {CKPT_DIR}

## 5 · Verify IoU

In [ ]:
import torch
from data.pets_dataset import create_dataloaders
from models.localization import VGG11Localizer

CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

model = VGG11Localizer(dropout_p=0.5, image_size=224).to(device)
model.load_state_dict(torch.load(f"{CKPT_DIR}/localizer.pth", map_location=device, weights_only=False))
model.eval()

def compute_iou(pred, gt, eps=1e-6):
    px1=pred[:,0]-pred[:,2]/2; py1=pred[:,1]-pred[:,3]/2
    px2=pred[:,0]+pred[:,2]/2; py2=pred[:,1]+pred[:,3]/2
    tx1=gt[:,0]-gt[:,2]/2;     ty1=gt[:,1]-gt[:,3]/2
    tx2=gt[:,0]+gt[:,2]/2;     ty2=gt[:,1]+gt[:,3]/2
    ix1=torch.max(px1,tx1); iy1=torch.max(py1,ty1)
    ix2=torch.min(px2,tx2); iy2=torch.min(py2,ty2)
    inter=(ix2-ix1).clamp(0)*(iy2-iy1).clamp(0)
    union=(px2-px1).clamp(0)*(py2-py1).clamp(0)+(tx2-tx1).clamp(0)*(ty2-ty1).clamp(0)-inter
    return inter/(union+eps)

n50, n75, total = 0, 0, 0
with torch.no_grad():
    for batch in test_loader:
        pred = model(batch["image"].to(device)).cpu()
        iou  = compute_iou(pred, batch["bbox"])
        n50  += (iou >= 0.50).sum().item()
        n75  += (iou >= 0.75).sum().item()
        total += len(iou)

print(f"\n✅  Accuracy @ IoU≥0.50 = {100*n50/total:.1f}%  (need > 60% for 4.2a)")
print(f"   Accuracy @ IoU≥0.75 = {100*n75/total:.1f}%  (need > 40% for 4.2b)")

## 6 · Get Drive File ID + Make Public

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_svc = build('drive', 'v3')

results = drive_svc.files().list(
    q="name='localizer.pth' and trashed=false",
    fields="files(id, name)"
).execute()

files = results.get('files', [])
if not files:
    print("❌  localizer.pth not found in Drive — did training finish?")
else:
    LOCALIZER_DRIVE_ID = files[0]['id']
    drive_svc.permissions().create(
        fileId=LOCALIZER_DRIVE_ID,
        body={'type': 'anyone', 'role': 'reader'}
    ).execute()
    print(f"✅  LOCALIZER_DRIVE_ID = {LOCALIZER_DRIVE_ID}")

## 7 · Update `multitask.py` + Push to GitHub

In [ ]:
import re

multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()

content = re.sub(
    r'LOCALIZER_DRIVE_ID\s*=\s*"[^"]*"',
    f'LOCALIZER_DRIVE_ID  = "{LOCALIZER_DRIVE_ID}"',
    content
)

with open(multitask_path, "w") as f:
    f.write(content)

!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"
!git add models/multitask.py
!git commit -m "task2: set LOCALIZER_DRIVE_ID for Gradescope"
!git push

print("\n🎉  Done! Submit to Gradescope to check 4.2a / 4.2b")